# Seasonal (airline-style) synthetic series vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts from a **seasonal ARIMA** (“airline” family) to **TimesFM** on synthetic **non-stationary seasonal** data.

**Why longer history before the first forecast:** a monthly **seasonal period** $m=12$ and the usual **airline** specification **SARIMA(0,1,1)(0,1,1)₁₂** need more than 11 points to be estimable. So here **$n=120$** and the first test index is **$k=37$** (history length 37 before predicting $y_{37}$). Test indices are **$k=37,\ldots,119$** (**83** one-step errors).

**DGP (readable):** integrated noise with a **seasonal MA** shock at lag 12 (same spirit as multiplicative seasonal models in levels):

$y_t = y_{t-1} + \varepsilon_t + \theta \, \varepsilon_{t-12}$ for $t \ge 12$, with i.i.d. standard normal $\varepsilon$, fixed $\theta$, and a random walk–style start for the first year.

**Classical:** **SARIMAX** with **order=(0,1,1)**, **seasonal_order=(0,1,1,12)** — the standard **airline** model on each expanding window.

## 1. Setup

**Hugging Face:** `HF_TOKEN` — `pip install -e ".[experiments]"`, `.env`, or `export HF_TOKEN=...`.

Imports, repo path, `load_dotenv`, **random seed**.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass
sys.path.insert(0, str(REPO / "src"))

import timesfm
from statsmodels.tsa.statespace.sarimax import SARIMAX

RNG_SEED = 42
N = 120
S = 12
THETA_SEASONAL = 0.4
K_FIRST = 37
rng = np.random.default_rng(RNG_SEED)

/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Simulate seasonal integrated series

Draw a long **ε** path. Seed **y[0], …, y[11]**, then for each **t ≥ 12** update **y[t]** from **y[t−1]**, **ε[t]**, and **ε[t−12]** in a simple loop.

In [2]:
def simulate_seasonal_airline(n: int, s: int, theta: float, rng: np.random.Generator) -> np.ndarray:
    eps = rng.normal(0.0, 1.0, n + s)
    y = np.zeros(n, dtype=np.float64)
    y[:s] = np.cumsum(rng.normal(0.0, 0.4, s))
    for t in range(s, n):
        y[t] = y[t - 1] + eps[t] + theta * eps[t - s]
    return y


y = simulate_seasonal_airline(N, S, THETA_SEASONAL, rng)

## 3. One-step “airline” forecast (classical)

Fit **SARIMAX(0,1,1)(0,1,1)₁₂** on `history` and return the one-step forecast.

In [3]:
def forecast_airline(history: np.ndarray, seasonal_period: int) -> float | None:
    try:
        if len(history) < seasonal_period + 10:
            return None
        res = SARIMAX(
            history.astype(np.float64),
            order=(0, 1, 1),
            seasonal_order=(0, 1, 1, seasonal_period),
        ).fit(disp=False, maxiter=200)
        return float(np.asarray(res.forecast(steps=1)).ravel()[0])
    except Exception:
        raise Exception

## 4. Load TimesFM (once)

**horizon = 1** in the loop.

In [4]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Expanding-window loop and MSE

For each **k = K_FIRST, …, N−1**: actual $= y[k]$, history $= y[:k]$. Compare **airline SARIMAX** vs **TimesFM**.

In [5]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))
se_air: list[float] = []
se_tf: list[float] = []
n_fail = 0

for k in test_ks:
    actual = float(y[k])
    hist = y[:k].astype(np.float32)
    pred_tf = float(tfm.forecast(horizon=1, inputs=[hist])[0][0, 0])
    pred_air = forecast_airline(y[:k], S)
    e_air = actual - pred_air
    e_tf = actual - pred_tf
    se_air.append(e_air**2)
    se_tf.append(e_tf**2)

rows.append(
    {
        "n": N,
        "seasonal_period": S,
        "k_first": K_FIRST,
        "n_test": len(test_ks),
        "airline_fit_failures": n_fail,
        "mse_airline_sarimax": float(np.mean(se_air)) if se_air else float("nan"),
        "mse_timesfm": float(np.mean(se_tf)) if se_tf else float("nan"),
    }
)

results = pd.DataFrame(rows)
results

,n,seasonal_period,k_first,n_test,airline_fit_failures,mse_airline_sarimax,mse_timesfm
0,120,12,37,83,0,0.915665,0.811261


## 6. Save table

CSV under `output/`.

In [6]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "seasonal_airline_vs_timesfm_results.csv"
results.to_csv(path, index=False)
print(path)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh_experiments/output/seasonal_airline_vs_timesfm_results.csv
